# 9 - Analyse CV Results

Script to evaluate the results of cross-validation training using multiple Foundation Model features with different ML models

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import seaborn as sns


In [ ]:
import time
from utils import print_time_since
start_time = time.perf_counter()

In [ ]:

pd.set_option("display.precision", 3)

In [ ]:
def getBestModelsForMetric(label,best_scores,metric='RMSE',segment='PanCK+'):
    #print(best_scores)
    #print(segment)
    combo_counts = best_scores[['Feature_Extractor','ML_Model']].value_counts()
    #print(combo_counts)
    #print(combo_counts)
    topcombo = combo_counts.index[0][0]+'-'+combo_counts.index[0][1]
    if len(combo_counts)==1:
        runnerup = topcombo
    else:
        runnerup = combo_counts.index[1][0]+'-'+combo_counts.index[1][1]
    return {
        'Segment':segment,
        'Label': label,
        'Metric': metric,
        'Top_Feature_Extractor': best_scores['Feature_Extractor'].value_counts().idxmax(),
        'Top_ML_Model': best_scores['ML_Model'].value_counts().idxmax(),
        'Top_Combination': topcombo,
        'RunnerUp_Combination': runnerup,
    }

In [ ]:
def getBestModels(label, scores, segment='PanCK+', groupbyfield='Gene'):
    metric_for_best='nRMSE'
    per_best = scores.loc[scores.groupby(groupbyfield)[metric_for_best].idxmin(),:]
    best_sig_nrmse = getBestModelsForMetric(label,per_best,metric_for_best,segment)
    
    metric_for_best='R2'
    per_best = scores.loc[scores.groupby(groupbyfield)[metric_for_best].idxmin(),:]
    best_sig_r2 = getBestModelsForMetric(label,per_best,metric_for_best,segment)
    
    metric_for_best='PearsonCorr'
    per_best = scores.loc[scores.groupby(groupbyfield)[metric_for_best].idxmax(),:]
    best_sig_corr = getBestModelsForMetric(label,per_best,metric_for_best,segment)

    #metric_for_best='ConcordanceCorr'
    #per_best = scores.loc[scores.groupby(groupbyfield)[metric_for_best].idxmax(),:]
    #best_sig_ccc = getBestModelsForMetric(label,per_best,metric_for_best,segment)

    return [best_sig_nrmse, best_sig_r2, best_sig_corr] #,best_sig_ccc]


In [ ]:
def get_mean_performance_summary(label, scores, segment='PanCK+', metric='nRMSE'):
    subset = scores[scores['Segment'] == segment]
    return (
        subset.groupby(['Feature_Extractor', 'ML_Model'])[metric]
        .mean()
        .sort_values()
        .reset_index()
    )

In [ ]:

root_path = "../data"
outcome_path = os.path.join(root_path,'02-input','outcomes')
results_path = os.path.join(root_path,'03-results_cv')
summary_path = os.path.join(root_path,'03-results_cv','summary')
table_path = os.path.join(root_path,'tables')

In [ ]:
min_r2 = 0.2
fe_order = ['Prov-Gigapath', 'UNI', 'Virchow']  
seg_order = ['CD45+', 'PanCK+']  

In [ ]:
contamination_genes = ['IGHG3',
 'IGHG1',
 'IGHG4',
 'IGHG2',
 'IGKC',
 'IGLL5',
 'IGHM',
 'IGHA1',
 'JCHAIN',
 'MZB1']

In [ ]:

panck_stds = pd.read_csv(os.path.join(outcome_path,"allgenes_panck_test_std.csv"))
cd45_stds = pd.read_csv(os.path.join(outcome_path,"allgenes_cd45_test_std.csv"))

Analyse results of ML models

In [ ]:
panck_stds["Segment"] = "PanCK+"
cd45_stds["Segment"] = "CD45+"
all_stds = pd.concat([panck_stds, cd45_stds], ignore_index=True)

# Data load

In [ ]:
per_gene_results = pd.read_csv(os.path.join(results_path,'allgenescores.csv'),index_col=0)


Remove contamination genes from PanCK

In [ ]:
cond = per_gene_results['Segment']=='PanCK+'
cond = cond & (per_gene_results.isin(contamination_genes).any(axis=1))
per_gene_results = per_gene_results[~cond]


In [ ]:
per_gene_results = per_gene_results.merge(all_stds, on=["Gene", "Segment"], how='left')
per_gene_results['nRMSE'] = per_gene_results['RMSE'] / per_gene_results['std']
#assert(per_gene_results['std'].isna().any() == False)

In [ ]:
metric='nRMSE'

# Get Best Models

elastic net failed to converge in a large percentage of models so we will remove it from evaluation now

Inspect best models based on those with R2 over threshold

## Panels

In [ ]:
panel_scores = per_gene_results[(per_gene_results['GeneSet']=='Panel') & (per_gene_results['R2'] > min_r2)]
panel_scores = panel_scores[~(panel_scores['ML_Model']=='elasticnet')]
panel_cd45 = panel_scores.loc[panel_scores['Segment'].str.contains('CD45+'),:].sort_values(by=metric)
panel_panck = panel_scores.loc[panel_scores['Segment'].str.contains('PanCK+'),:].sort_values(by=metric)

check top model combinations

In [ ]:
best_panel_cd45 = getBestModels('Panel',panel_cd45,segment='CD45+')
best_panel_panck = getBestModels('Panel',panel_panck,segment='PanCK+')

## HVGs


In [ ]:
hvg_scores = per_gene_results[(per_gene_results['GeneSet']=='HVG')& (per_gene_results['R2'] > min_r2)]
hvg_scores = hvg_scores[~(hvg_scores['ML_Model']=='elasticnet')]
hvg_cd45 = hvg_scores.loc[hvg_scores['Segment'].str.contains('CD45+'),:].sort_values(by=metric)
hvg_panck = hvg_scores.loc[hvg_scores['Segment'].str.contains('PanCK+'),:].sort_values(by=metric)

In [ ]:
best_hvg_cd45 = getBestModels('HVG',hvg_cd45,segment='CD45+')
best_hvg_panck = getBestModels('HVG',hvg_panck,segment='PanCK+')

## SVGs

In [ ]:

svg_scores = per_gene_results[(per_gene_results['GeneSet']=='SVG') & (per_gene_results['R2'] > min_r2)]
svg_scores = svg_scores[~(svg_scores['ML_Model']=='elasticnet')]
svg_cd45 = svg_scores.loc[svg_scores['Segment'].str.contains('CD45+'),:]
svg_panck = svg_scores.loc[svg_scores['Segment'].str.contains('PanCK+'),:]
best_svg_cd45 = getBestModels('SVG',svg_cd45,segment='CD45+')
best_svg_panck = getBestModels('SVG',svg_panck,segment='PanCK+')


## Aggregate

In [ ]:
best_cd45 = pd.DataFrame(best_panel_cd45+best_hvg_cd45+best_svg_cd45).sort_values(by='Metric')
best_cd45

In [ ]:
best_panck = pd.DataFrame(best_panel_panck+best_hvg_panck+best_svg_panck).sort_values(by='Metric')
best_panck

# Examine Results

including elastic net here
How many genes have R2 greater than 0.2, 0.3, 0.4

In [ ]:
def count_over_thresh(df, thresh):
    # Count unique genes per GeneSet
    return (
        df[df['R2']>thresh]
        .groupby(["Segment","GeneSet"])["Gene"]
        .nunique()
        .reset_index(name=f"N_over_{thresh}")
    )


In [ ]:
over02 = count_over_thresh(per_gene_results, 0.2)
over03 = count_over_thresh(per_gene_results, 0.3)
over04 = count_over_thresh(per_gene_results, 0.4)

In [ ]:
r2_over = pd.merge(over02,over03).merge(over04,how='left').fillna(0)
r2_over

# Check Mean Performance

In [ ]:
robust_results = per_gene_results[per_gene_results['R2']>min_r2]

## All Results

In [ ]:
grouped = (
    per_gene_results
    .groupby(["Segment", "Feature_Extractor", "ML_Model"])[["nRMSE","R2", "PearsonCorr"]]
    .mean()
    .reset_index()
)
best_per_group = (
    grouped
    .sort_values("nRMSE")
    .groupby(["Segment"])
    .first()
    .reset_index()
)

display(best_per_group)


In [ ]:
grouped_robust = (
    robust_results
    .groupby(["Segment", "Feature_Extractor", "ML_Model"])[["nRMSE","R2", "PearsonCorr"]]
    .mean()
    .reset_index()
)
best_per_group_robust = (
    grouped_robust
    .sort_values("nRMSE")
    .groupby(["Segment"])
    .first()
    .reset_index()
)

display(best_per_group_robust)

In [ ]:
grouped_plus = grouped.join(grouped_robust, on=None, how='left', rsuffix='_robust').drop(columns=['Segment_robust','Feature_Extractor_robust','ML_Model_robust'])
grouped_plus

### CD45

In [ ]:
#CD45 only - check top 2
mean_nrmse = get_mean_performance_summary('all_cd45',per_gene_results, segment='CD45+', metric='nRMSE')
mean_nrmse.head(4)

uni is better than virchow mraginally


In [ ]:
min_nrmse_cd45 = round(mean_nrmse.nRMSE.min(),2)
print(f"Minimum nRMSE for CD45+ segments is {min_nrmse_cd45} using {mean_nrmse['Feature_Extractor'][0]}-{mean_nrmse['ML_Model'][0]}")

### PanCK

In [ ]:
mean_nrmse = get_mean_performance_summary('all_panck',per_gene_results, segment='PanCK+', metric='nRMSE')
mean_nrmse.head(2)

In [ ]:
min_nrmse_panck = round(mean_nrmse.nRMSE.min(),2)
print(f"Minimum nRMSE for PanCK+ segments is {min_nrmse_panck} using {mean_nrmse['Feature_Extractor'][0]}-{mean_nrmse['ML_Model'][0]}")

In [ ]:
robust_results =robust_results.drop(columns=['source'])

# boxplots

In [ ]:
robust_results.to_csv(os.path.join(summary_path,'goodpredictors.csv'))

In [ ]:

segment_names={'CD45+':'Immune Regions (CD45⁺)', 'PanCK+':'Tumour Regions (PanCK⁺)'}


In [ ]:
robust_results_formatted = robust_results.copy()
per_gene_results_formatted = per_gene_results.copy()

In [ ]:
robust_results_formatted.loc[:,'Feature_Extractor'] = robust_results_formatted['Feature_Extractor'].replace({
    'uni': 'UNI',
    'gigapath': 'Prov-Gigapath',
    'virchow': 'Virchow',
    'fused': 'Fused',
})
robust_results_formatted.loc[:,'ML_Model'] = robust_results_formatted['ML_Model'].replace({
    'elasticnet': 'Elastic Net',
    'randomforest': 'Random Forest',
    'gradboost': 'Gradient Boosted Trees',
    'SVM': 'Support Vector Machine'
})

In [ ]:
per_gene_results_formatted.loc[:,'Feature_Extractor'] = per_gene_results_formatted['Feature_Extractor'].replace({
    'uni': 'UNI',
    'gigapath': 'Prov-Gigapath',
    'virchow': 'Virchow',
    'fused': 'Fused',
})
per_gene_results_formatted.loc[:,'ML_Model'] = per_gene_results_formatted['ML_Model'].replace({
    'elasticnet': 'Elastic Net',
    'randomforest': 'Random Forest',
    'gradboost': 'Gradient Boosted Trees',
    'SVM': 'Support Vector Machine'
})

In [ ]:
per_gene_results_formatted.head(2)

## Significance Test

In [ ]:
import itertools
from scipy.stats import mannwhitneyu
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

In [ ]:

def compare_groups_paired(df, metric='nRMSE', alternative='two-sided',
                          zero_method='pratt', fdr_method='fdr_bh'):
    """
    Pairwise method comparison within each Segment using Wilcoxon signed-rank.
    Assumes df has one row per (Segment, Gene, ML_Model, Feature_Extractor, metric).
    If multiple rows per gene-method exist, their metric is averaged.
    """
    required = {'Segment', 'Gene', 'ML_Model', 'Feature_Extractor', metric}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    records = []

    for segment, seg in df.groupby('Segment'):
        # wide table: index=Gene, columns=(ML_Model, Feature_Extractor), values=metric
        wide = (seg
                .groupby(['Gene', 'ML_Model', 'Feature_Extractor'])[metric]
                .mean()
                .unstack(['ML_Model', 'Feature_Extractor']))

        methods = list(wide.columns)
        if len(methods) < 2:
            continue

        for m1, m2 in itertools.combinations(methods, 2):
            pair = wide[[m1, m2]].dropna()
            if pair.empty:
                continue

            x = pair[m1].to_numpy()
            y = pair[m2].to_numpy()
            d = x - y

            # All ties -> no difference
            if np.allclose(d, 0):
                stat, p = 0.0, 1.0
            else:
                stat, p = wilcoxon(x, y, alternative=alternative, zero_method=zero_method)

            med_diff = float(np.median(d))  # <0 means m1 < m2 (e.g., better if lower is better)

            records.append({
                'Segment': segment,
                'Group 1': m1,           # tuple: (ML_Model, Feature_Extractor)
                'Group 2': m2,
                'W-stat': stat,          # Wilcoxon signed-rank statistic
                'p-value': p,
                'n_pairs': int(len(d)),
                'median_diff': med_diff
            })

    if not records:
        return pd.DataFrame(columns=['Segment','Group 1','Group 2','W-stat','p-value',
                                     'q-value','n_pairs','median_diff','significant-p','significant-q'])

    out = pd.DataFrame(records)

    # FDR within each Segment
    out['q-value'] = np.nan
    for seg, idx in out.groupby('Segment').groups.items():
        pvals = out.loc[idx, 'p-value'].values
        _, q, _, _ = multipletests(pvals, method=fdr_method)
        out.loc[idx, 'q-value'] = q

    out['q-value'] = out['q-value'].astype(float).round(4)
    out['significant-p'] = out['p-value'] < 0.05
    out['significant-q'] = out['q-value'] < 0.05

    # sort by adjusted p
    return out.sort_values(['Segment','q-value','p-value'])


In [ ]:
def compare_groups(df, metric='nRMSE',group_fields=['ML_Model', 'Feature_Extractor']):

    #print("comparing:", group_fields)
    # Collect all results
    all_results = []

    # Loop through each segment
    for segment, seg_data in df.groupby('Segment'):
        # Get all unique (model, extractor) pairs in this segment
        groups = seg_data.groupby(group_fields)

        # List of group names and corresponding nRMSEs
        group_names = list(groups.groups.keys())
        #print(group_names)
        group_data = {name: groups.get_group(name)[metric] for name in group_names}
    
        # Perform pairwise tests
        for (name1, name2) in itertools.combinations(group_names, 2):
            data1 = group_data[name1]
            data2 = group_data[name2]
            stat, p = mannwhitneyu(data1, data2, alternative='two-sided')

            all_results.append({
                'Segment': segment,
                'Group 1': name1,
                'Group 2': name2,
                'p-value': p
            })

    # Convert to DataFrame
    signif_test = pd.DataFrame(all_results)

    # Apply FDR correction within each segment
    signif_test['q-value'] = None
    for segment in signif_test['Segment'].unique():
        mask = signif_test['Segment'] == segment
        pvals = signif_test.loc[mask, 'p-value']
        _, corrected, _, _ = multipletests(pvals, method='fdr_bh')
        signif_test.loc[mask, 'q-value'] = corrected

    # Format for clarity
    signif_test[['q-value']] = signif_test[['q-value']].astype(float).round(4)
    signif_test['significant-q'] = (signif_test['q-value']<0.05)
    signif_test['significant-p'] = (signif_test['p-value']<0.05)

    signif_test=signif_test.sort_values('q-value')
    return signif_test

# Compare
compare results of different feature_extractor/ml_model combinations to see if there is a statistically-significant difference

restrict to only the 'good' predictors (R2>0.2)

In [ ]:
signif_nrmse = compare_groups(robust_results, metric='nRMSE')
signif_nrmse.head()

compare groups without the concatenated features

In [ ]:

signif_nrmse = compare_groups(robust_results[~robust_results['Feature_Extractor'].isin(['fused'])], metric='nRMSE')
signif_nrmse.to_csv(os.path.join(summary_path,'goodpredictors_comparison_PLUS_Fused.csv'))
signif_nrmse.sort_values('significant-q')

In [ ]:
#wilcoxon_nocomb = compare_groups_paired(robust_results[~robust_results['Feature_Extractor'].isin(['fused'])], metric='nRMSE')
#wilcoxon_nocomb.to_csv(os.path.join(summary_path,'wilcoxon_NOcombined.csv'))

In [ ]:
#wilcoxon_comb = compare_groups_paired(robust_results)
#wilcoxon_comb.to_csv(os.path.join(summary_path,'wilcoxon_combined.csv'))

In [ ]:
signif_nrmse = compare_groups(robust_results_formatted, metric='nRMSE')
signif_nrmse_nocombined = compare_groups(robust_results_formatted[~robust_results_formatted['Feature_Extractor'].isin(['fused'])], metric='nRMSE')
signif_nrmse_nocombined.to_csv(os.path.join(summary_path,'goodpredictors_comparison.csv'))
signif_nrmse_nocombined.sort_values('significant-q',ascending=False)

In [ ]:
signif_nrmse_nocombined[signif_nrmse_nocombined['significant-p']]

### Compare at FM level

In [ ]:
compare_fms = compare_groups(per_gene_results_formatted, metric='nRMSE',group_fields=['Feature_Extractor'])
compare_fms

In [ ]:
compare_fms = compare_groups(robust_results_formatted, metric='nRMSE',group_fields=['Feature_Extractor'])
compare_fms


# Repeat without Elastic Net

Elastic Net suffered  convergence failure for many genes so it is not a suitable model to select if we are looking for a single ML model to use to train each gene

In [ ]:
t1 = print_time_since(start_time, "execution time to this point")

remove elastic net due to convergence issues

In [ ]:
sns.reset_defaults()

In [ ]:
#don't include combined
cond = (robust_results_formatted['ML_Model']=='Elastic Net')
df = robust_results_formatted[~cond].sort_values('ML_Model')

In [ ]:

sns.set_palette(sns.color_palette("colorblind"))


# Set up the FacetGrid to split by 'Segment'
g = sns.catplot(
    data=df,
    x='ML_Model',
    y='nRMSE',
    hue='Feature_Extractor',
    hue_order=fe_order,     
    col='Segment',
    kind='box',
    height=7,
    aspect=0.8,
    palette='viridis',
    col_order=seg_order,  # Ensure consistent order
)

# Add titles and formatting
#g.set_titles("Segment: {col_name}")
g.set_axis_labels("", "Normalised RMSE")
#g.set(ylim=(0.5, 1.2))  # Set y-axis range
g.legend.set_title("")#Feature Extractor")
g.legend.set_bbox_to_anchor((1.01, 0.85)), 
g.legend.set_loc('upper right')
g.legend.set_frame_on(True)
g.set_ylabels("Normalised RMSE", fontsize=16)
for ax, title in zip(g.axes.flat, g.col_names):
    ax.set_title(f"{segment_names[title]}", fontsize=20)
for ax in g.axes.flat:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right', fontsize=16)


# title
#g._legend.get_title().set_fontsize(14)

# each text entry
for txt in g._legend.get_texts():
    txt.set_fontsize(14)

## annotate significant comparisons
bar_offset = {
    fe: i for i, fe in enumerate(sorted(df['Feature_Extractor'].unique()))
}
for ax, segment in zip(g.axes.flat, g.col_names):
    seg_results = signif_nrmse_nocombined[signif_nrmse_nocombined['Segment'] == segment]
    data = df[df['Segment'] == segment]
    y_max = 1.2
    
    xticks = [tick.get_text() for tick in ax.get_xticklabels()]
    xpos = {
        (ml, fe): i + 0.2 * bar_offset[fe]
        for i, ml in enumerate(xticks)
        for fe in bar_offset
    }
    y_base = y_max + 0.01

    offset_y = 0.05  # vertical spacing between annotation lines
    height_map = {}  # to track used line heights per pair

    # Get base x-position for each ML_Model
    ml_models = sorted(df['ML_Model'].unique())
    model_to_index = {model: i for i, model in enumerate(ml_models)}

    # Compute per-hue offset (depends on number of extractors)
    extractors = sorted(df['Feature_Extractor'].unique())
    bar_width = 0.8 / len(extractors)  # Auto-dodge width
    extractor_offsets = {
        extractor: i * bar_width - (0.8 - bar_width) / 2
        for i, extractor in enumerate(extractors)
    }

    for _, row in seg_results.iterrows():
        q = row['q-value']
        if q < 0.01:
            marker = "**"  # Significant
        # elif q < 0.05:
        #     marker = "*"  # Significant
        # elif 0.05 <= q < 0.06:
        #     marker = "~"  # Almost significant
        else:
            continue  # Skip non-significant comparisons

        (ml1, fe1) = row['Group 1']
        (ml2, fe2) = row['Group 2']

        if ml1 in model_to_index and ml2=='Support Vector Machine': 
            x1 = model_to_index[ml1] + extractor_offsets[fe1]
            x2 = model_to_index[ml2] + extractor_offsets[fe2]

            line_col =  'black'
            if x1 > x2:
                x1, x2 = x2, x1

            pair_key = (round(x1, 3), round(x2, 3))
            line_y = height_map.get(pair_key, y_base + len(height_map) * offset_y)
            height_map[pair_key] = line_y

            ax.plot([x1, x1, x2, x2], [line_y, line_y + 0.01, line_y + 0.01, line_y],
                    lw=1.2, c=line_col)
            ax.text((x1 + x2) / 2, line_y + 0.012, marker, ha='center', va='bottom', color='black')

plt.tight_layout(rect=[0, 0.03, 1, 0.9])
#plt.suptitle("Comparison of performance across feature extractors and models", fontsize=24)
# Add the note at the bottom
plt.figtext(0.5, 0.01, '"**" indicates a adjusted p-value < 0.01', ha='center', fontsize=14)
plt.figtext(0.5, 0.01, '"*" indicates a adjusted p-value < 0.05', ha='center', fontsize=14)
#plt.figtext(0.5, 0.01, '"~" indicates a result approaching statistical significance', ha='center', fontsize=14)

#
plt.show()

# Model Selection

In [ ]:
en = {
    ('Elastic Net', 'Prov-Gigapath'),
    ('Elastic Net', 'UNI'),
    ('Elastic Net', 'Virchow'),
    ('Elastic Net', 'Fused'),
}
signif_nrmse_no_en = signif_nrmse[~signif_nrmse['Group 2'].isin(en)]
signif_nrmse_no_en = signif_nrmse_no_en[~signif_nrmse_no_en['Group 1'].isin(en)]

In [ ]:
suppvec = {
    ('Support Vector Machine', 'Prov-Gigapath'),
    ('Support Vector Machine', 'UNI'),
    ('Support Vector Machine', 'Virchow'),
    ('Support Vector Machine', 'Fused'),
}

SVM is superior

In [ ]:
signif_nrmse_no_en = signif_nrmse_no_en.sort_values('q-value')
signif_nrmse_no_en[signif_nrmse_no_en['significant-q']==True]

# SVM vs other models

In [ ]:
cond = (~signif_nrmse_no_en['Group 1'].isin(suppvec))&(signif_nrmse_no_en['Group 2'].isin(suppvec))
cond = cond & (signif_nrmse_no_en['significant-q'])
signif_nrmse_no_en[cond].sort_values(['q-value'])

In [ ]:

signif_nrmse_no_en[(signif_nrmse_no_en['Group 1'].isin(suppvec))&(~signif_nrmse_no_en['Group 2'].isin(suppvec))].sort_values(['q-value'])

We can see that, in CD45+ segments, SVM+combined is significantly better than SVM+gigapath (combined is also better than uni and virchow but not statistically significant). SVM+Virchow is statistically significantly better than SVM+gigapath and better (but not significantly) than UNI. So SVM+combined has the most support as the top model combination for CD45+

In PanCK+ segments none of the SVM comparisons are significant but combined achieves better performance than uni, gigapath and virchow. 

In [ ]:
full_fe_order = ['Prov-Gigapath', 'UNI', 'Virchow','Fused']

In [ ]:

# Set up the FacetGrid to split by 'Segment'
g = sns.catplot(
    data=robust_results_formatted,
    x='ML_Model',
    y='nRMSE',
    hue='Feature_Extractor',
    hue_order=full_fe_order,     
    col='Segment',
    kind='bar',
    height=5,
    aspect=1.2,
    palette='viridis',
    col_order=[ 'CD45+', 'PanCK+'],  # Ensure consistent order
)

# Add titles and formatting
#g.set_titles("Segment: {col_name}")
g.set_axis_labels("", "Normalised RMSE")
#g.set(ylim=(0.5, 1.2))  # Set y-axis range
g.legend.set_title("Feature Extractor")
g.legend.set_bbox_to_anchor((1, 1)), 
g.legend.set_loc('upper right')
g.legend.set_frame_on(True)
for ax, title in zip(g.axes.flat, g.col_names):
    ax.set_title(f"{segment_names[title]}")
for ax in g.axes.flat:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='center')



## annotate significant comparisons
bar_offset = {
    fe: i for i, fe in enumerate(sorted(robust_results_formatted['Feature_Extractor'].unique()))
}
for ax, segment in zip(g.axes.flat, g.col_names):
    data = robust_results_formatted[robust_results_formatted['Segment'] == segment]
    y_max = data['nRMSE'].max()
    
    xticks = [tick.get_text() for tick in ax.get_xticklabels()]
    xpos = {
        (ml, fe): i + 0.2 * bar_offset[fe]
        for i, ml in enumerate(xticks)
        for fe in bar_offset
    }
    y_base = y_max + 0.03
    offset_y = 0.03  # vertical spacing between annotation lines
    height_map = {}  # to track used line heights per pair


plt.suptitle("Comparison of error rate across feature extractors and models")
plt.tight_layout()
plt.show()

# Stats for Results - Performance per Model

Elastic net

should this be 'Fused' rather than 'fused'?

In [ ]:
unique_gene_results = per_gene_results[(per_gene_results['Feature_Extractor']=='fused')&
                (per_gene_results['ML_Model']=='SVM')]

In [ ]:
unique_gene_results['Segment'].value_counts()

In [ ]:
per_gene_results.groupby(['Feature_Extractor','ML_Model'])['Gene'].count()

should not hard code 237

In [ ]:
robust_counts =pd.DataFrame(robust_results.groupby(['Feature_Extractor','ML_Model'])['Gene'].count())
robust_counts['pct'] = (robust_counts['Gene'] / 237*100)
robust_counts

In [ ]:
robust_counts.describe()

In [ ]:
en_mean_robust = robust_results.loc[robust_results['ML_Model']=='elasticnet','Feature_Extractor'].value_counts().mean()
print(f"Elastic Net showed robust performance in an average of {en_mean_robust} genes per feature extractor")
print(f"{en_mean_robust/237:.2%}")

In [ ]:
robust_results.loc[robust_results['ML_Model']=='elasticnet','Feature_Extractor'].value_counts()

In [ ]:
robust_results_formatted.groupby(['Feature_Extractor'])[['nRMSE','R2','PearsonCorr']].mean()

In [ ]:
robust_results_formatted.groupby(['Feature_Extractor'])[['nRMSE','R2','PearsonCorr']].agg(['mean','median','min', 'max'])

In [ ]:
per_gene_results_formatted.groupby(['Segment','Feature_Extractor'])[['nRMSE','PearsonFDR']].agg(['mean','median','min', 'max'])

In [ ]:
robust_results_formatted.groupby(['Segment','Feature_Extractor'])[['nRMSE','R2','PearsonCorr']].mean()

In [ ]:
results_table = per_gene_results_formatted.groupby(['Segment','Feature_Extractor','ML_Model'])[['R2','nRMSE','PearsonCorr']].mean(numeric_only=True)
results_table

In [ ]:

robust_table = robust_results_formatted.groupby(['Segment','Feature_Extractor','ML_Model'])[['R2','nRMSE','PearsonCorr']].mean(numeric_only=True)
#robust_results_formatted.groupby(['Segment','Feature_Extractor','ML_Model'])[['R2','nRMSE','PearsonCorr',"ConcordanceCorr"]].mean(numeric_only=True)
robust_table

In [ ]:
results_table.to_csv(os.path.join(table_path,'results_table_allgenes.csv'))
robust_table.to_csv(os.path.join(table_path,'results_table_robust.csv'))

LaTeX code for Overleaf

In [ ]:
print(robust_table.to_latex(caption="caption here",float_format="{:0.3f}".format))


## Add Significance

# Save Best Predictors

drop elastic net at this point

In [ ]:
robust_results_no_en = robust_results[~(robust_results['ML_Model']=='elasticnet')]

In [ ]:
good_grouped = (
    robust_results_no_en
    .groupby(["Segment", "GeneSet", "Feature_Extractor", "ML_Model"])[["nRMSE","R2","PearsonCorr", "SpearmanCorr"]]
    .mean()
    .reset_index()
)

best per geneset based on R2

In [ ]:
best_per_good_group = (
    good_grouped
    .sort_values("nRMSE",ascending=True)
    .groupby(["Segment", "GeneSet"])
    .first()
    .reset_index()
)

#best_per_good_group.to_csv(os.path.join(summary_path,'best_model_per_group_goodpredictors.csv'), index=False)
best_per_good_group

## Select one model combination per segment

top 3 models in each segment. 5 out of 6 are SVM

In [ ]:
good_grouped_seg = (
    robust_results_no_en
    .groupby(["Segment","Feature_Extractor", "ML_Model"])[["nRMSE","R2","PearsonCorr", "SpearmanCorr"]]
    .mean()
    .reset_index()
)
good_grouped_seg.sort_values("nRMSE").groupby(["Segment"]).head(2)

In [ ]:
good_grouped_seg = (
    robust_results_no_en
    .groupby(["Segment","Feature_Extractor", "ML_Model"])[["nRMSE","R2","PearsonCorr", "SpearmanCorr"]]
    .mean()
    .reset_index()
)
good_grouped_seg.sort_values("R2",ascending=False).groupby(["Segment"]).head(2)

In [ ]:
good_grouped_seg = (
    robust_results_no_en
    .groupby(["Segment","Feature_Extractor", "ML_Model"])[["nRMSE","R2","PearsonCorr", "SpearmanCorr"]]
    .mean()
    .reset_index()
)
good_grouped_seg.sort_values("PearsonCorr",ascending=False).groupby(["Segment"]).head(2)

In [ ]:
good_grouped_seg = (
    robust_results_no_en[~(robust_results_no_en['Feature_Extractor']=='fused')]
    .groupby(["Segment","Feature_Extractor", "ML_Model"])[["nRMSE","R2","PearsonCorr", "SpearmanCorr"]]
    .mean()
    .reset_index()
)

based on nRMSE

In [ ]:
best_per_good_segment = (
    good_grouped_seg
    .sort_values("nRMSE")
    .groupby(["Segment"])
    .first()
    .reset_index()
)
best_per_good_segment

We will also include combined+SVM for both - it is the single best overall for CD45+

Out of all the combined feature models, combined+SVM was the best

In [ ]:
good_grouped_seg = (
    robust_results_no_en[(robust_results_no_en['Feature_Extractor']=='fused')]
    .groupby(["Feature_Extractor", "ML_Model"])[["nRMSE","R2","PearsonCorr", "SpearmanCorr"]]
    .mean()
    .reset_index()
)
good_grouped_seg.sort_values("nRMSE", ascending=True)

In [ ]:
best_per_good_segment_combined = (
    good_grouped_seg
    .sort_values("nRMSE")
    .groupby(["Segment"])
    .first()
    .reset_index()
)
best_per_good_segment_combined

In [ ]:
best_per_good_segment = pd.concat([best_per_good_segment,best_per_good_segment_combined])
best_per_good_segment

In [ ]:
best_per_good_segment.to_csv(os.path.join(summary_path,'best_model_per_segment_goodpredictors.csv'), index=False)

# END